# M13 Lab — Agentic DS Workflows

**Datasets:** `housing_prices.csv`, `ecommerce_orders.csv` &nbsp;|&nbsp; **Focus:** ReAct loop, tool design, verification


In [ ]:
MODULE_ID = "M13"
# ── Confusion reporter ───────────────────────────────────────────────────────
# Run this cell once to set up the reporter, then call it any time you are
# confused about a term or concept. It logs the entry to the instructor
# dashboard so they can address common pain-points.
#
#   Usage (in any later cell):
#       await im_confused("overfitting")
#       await im_confused("gradient descent", "not sure how the learning rate affects convergence")

import json as _json

async def im_confused(term: str, note: str = ""):
    """Report a confusing term to your instructor.
    
    Args:
        term: the word / concept that confused you (e.g. "overfitting")
        note: optional extra detail (e.g. "what does the bias-variance tradeoff mean here?")
    """
    try:
        from pyodide.http import pyfetch
        payload = {
            "word": term,
            "message": note,
            "source": "self_report",
            "moduleId": MODULE_ID,
        }
        resp = await pyfetch(
            "/api/error-log",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps(payload),
            credentials="include",
        )
        if resp.ok:
            print(f"✅ Reported \"{term}\" — your instructor will see this in the Confusion dashboard.")
        else:
            print(f"⚠️  Could not report (HTTP {resp.status}). Are you logged in to DataPath?")
    except ImportError:
        # Running outside JupyterLite (e.g. plain Jupyter / Docker)
        import requests as _req
        payload = {"word": term, "message": note, "source": "self_report", "moduleId": MODULE_ID}
        try:
            r = _req.post("http://localhost:3001/api/error-log", json=payload, timeout=5)
            print("✅ Reported!" if r.ok else f"⚠️  HTTP {r.status_code}")
        except Exception as e:
            print(f"⚠️  Could not reach DataPath server: {e}")
    except Exception as e:
        print(f"⚠️  Unexpected error: {e}")

print("✓ Confusion reporter ready.")
print("  Usage: await im_confused(\"term you found confusing\")")


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def load_csv(path: str) -> pd.DataFrame:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    return pd.read_csv(file_path)

def describe_dataframe(path: str) -> dict:
    df = load_csv(path)
    return {
        "shape": df.shape,
        "columns": list(df.columns),
        "missing": df.isna().sum().to_dict(),
        "numeric_summary": df.describe(include="number").to_dict(),
    }

def plot_histogram(path: str, column: str) -> dict:
    df = load_csv(path)
    if column not in df.columns:
        raise KeyError(f"Column not found: {column}")
    ax = df[column].dropna().plot(kind="hist", bins=20, title=f"Histogram of {column}")
    plt.show()
    return {
        "column": column,
        "mean": float(df[column].mean()),
        "median": float(df[column].median()),
        "non_null": int(df[column].notna().sum()),
    }


In [ ]:
import json as _json, re

async def call_ollama(messages, model="gemma4:e2b"):
    """Call Ollama chat API. Works in JupyterLite (pyfetch) and Docker (requests)."""
    try:
        from pyodide.http import pyfetch
        resp = await pyfetch(
            "http://localhost:11434/api/chat",
            method="POST",
            headers={"Content-Type": "application/json"},
            body=_json.dumps({"model": model, "messages": messages, "stream": False}),
        )
        data = await resp.json()
    except ImportError:
        import requests
        r = requests.post("http://localhost:11434/api/chat",
                          json={"model": model, "messages": messages, "stream": False}, timeout=120)
        r.raise_for_status()
        data = r.json()
    return data["message"]["content"]

def parse_action(text: str):
    final_match = re.search(r"Final Answer:\s*(.*)", text, flags=re.S)
    if final_match:
        return {"type": "final", "content": final_match.group(1).strip()}
    action_match = re.search(r"Action:\s*(\w+)\nAction Input:\s*(.*?)(?=\nObservation:|$)", text, flags=re.S)
    if action_match:
        return {"type": "action", "tool": action_match.group(1).strip(), "input": action_match.group(2).strip()}
    return {"type": "unknown", "content": text}

print("✓ ReAct helpers loaded. Note: call_ollama is async — use: result = await call_ollama(...)")


In [ ]:
import json as _json

async def run_agent(goal: str, tools: dict, max_iters: int = 4):
    """Minimal ReAct loop. Degrades gracefully: returns a 'Stopped' result
    instead of raising if the local model misbehaves (small models often do)."""
    tool_docs = "\n".join(f"- {name}: {fn.__doc__ or 'no docstring'}" for name, fn in tools.items())
    messages = [
        {"role": "system", "content": (
            "You are a ReAct data-science agent. Think briefly, use one tool at a time, "
            "and reply in exactly one of these formats:\n"
            "Action: <tool_name>\nAction Input: <JSON object>\n"
            "OR\nFinal Answer: <concise grounded answer>\n\n"
            f"Available tools:\n{tool_docs}")},
        {"role": "user", "content": goal},
    ]
    trace = []
    for _ in range(max_iters):
        agent_text = await call_ollama(messages)
        trace.append(agent_text)
        decision = parse_action(agent_text)
        if decision["type"] == "final":
            return {"final_answer": decision["content"], "trace": trace}
        if decision["type"] != "action":
            return {"final_answer": "Stopped: could not parse an action.", "trace": trace}
        tool_name = decision["tool"]
        if tool_name not in tools:
            return {"final_answer": f"Stopped: unknown tool {tool_name!r}.", "trace": trace}
        try:
            tool_input = _json.loads(decision["input"])
        except Exception:
            tool_input = {}
        try:
            observation = tools[tool_name](**tool_input)
        except Exception as e:
            observation = f"Tool error: {e}"
        messages.append({"role": "assistant", "content": agent_text})
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    return {"final_answer": "Stopped: max iterations reached.", "trace": trace}

print("✓ run_agent ready (async, graceful).")


In [ ]:
tools = {
    "describe_dataframe": describe_dataframe,
    "plot_histogram": plot_histogram,
}

result = await run_agent(
    goal=(
        "Explore housing_prices.csv. First inspect the dataset structure. "
        "Then decide whether one histogram would help. "
        "End with Final Answer containing 2 grounded findings."
    ),
    tools=tools,
    max_iters=4,
)

print("FINAL ANSWER:\n", result["final_answer"])
print("\nTRACE:")
for item in result["trace"]:
    print("-" * 60)
    print(item)


## L13.5 [AI-OFF] — Verification exercise

Without using any AI help, inspect the agent trace from the previous cell.
In a fresh code or markdown cell, do all of the following:
1. identify one claim that needs manual recomputation,
2. recompute it directly from the dataset,
3. state whether the agent's wording was justified,
4. name one improvement you would make to the toolset or stopping rules.
